# Whiteboard video trên Google Colab

Notebook này chạy dự án mã nguồn mở [chalkboard](https://github.com/Atharva-Kanherkar/chalkboard) trên máy ảo Colab. Máy Windows của bạn không phải cài model hay thư viện.

Mặc định notebook dùng **Qwen2.5 3B qua Ollama** để viết kịch bản và **Piper tiếng Việt** để đọc lời thoại. Hai model chạy trong phiên Colab; không cần API key. Hình ảnh AI trả phí được tắt, nhưng các sơ đồ nét vẽ, chữ, phụ đề và nhạc nền vẫn có thể được render.

> Thời gian cài lần đầu có thể lâu vì Colab tải Chromium, model và giọng đọc. Máy ảo Colab có thể bị xóa khi phiên kết thúc; hãy tải MP4 về sau khi chạy.

## 1. Cài công cụ hệ thống

Chạy cell này một lần trong mỗi phiên Colab. Runtime cần Linux.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y curl ca-certificates ffmpeg sox zstd
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get -qq install -y nodejs
!npm install -g pnpm
!node --version
!pnpm --version

## 2. Lấy source và cài renderer

Notebook clone repo công khai từ GitHub vào máy ảo Colab. Chromium được cài cùng các thư viện Linux mà Playwright yêu cầu.

In [ ]:
import os
import subprocess

REPO_DIR = "/content/chalkboard"
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Atharva-Kanherkar/chalkboard.git",
        REPO_DIR
    ], check=True)

%cd /content/chalkboard
!pnpm install
!pnpm --filter @chalkboard/renderer exec playwright install --with-deps chromium

## 3. Cài model viết kịch bản miễn phí

Qwen2.5 3B tải khoảng 1,9 GB. Ollama sẽ chạy trong máy ảo Colab. Nếu Colab cấp GPU, Ollama có thể dùng GPU; nếu không, nó chạy bằng CPU và chậm hơn.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import time
import urllib.request

OLLAMA_LOG = open("/tmp/ollama.log", "w")
OLLAMA_PROCESS = subprocess.Popen(
    ["ollama", "serve"],
    stdout=OLLAMA_LOG,
    stderr=subprocess.STDOUT
)

for attempt in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama chưa khởi động. Xem /tmp/ollama.log")

!ollama pull qwen2.5:3b
os.environ["OLLAMA_MODEL"] = "qwen2.5:3b"

## 4. Cài giọng đọc tiếng Việt miễn phí

Piper tải model giọng đọc tiếng Việt vào máy ảo. Không cần tải file model về máy tính.

In [ ]:
!python -m pip -q install piper-tts
!mkdir -p /content/piper-voices
%cd /content/piper-voices
!python -m piper.download_voices vi_VN-vais1000-medium
%cd /content/chalkboard

os.environ["PIPER_MODEL"] = "/content/piper-voices/vi_VN-vais1000-medium.onnx"

## 5. Tạo video

Nhập chủ đề hoặc yêu cầu cụ thể. Kịch bản và thuyết minh sẽ dùng tiếng Việt. Video được xuất ở định dạng MP4 và tải về trình duyệt khi hoàn tất.

In [ ]:
topic = input("Nhập chủ đề video: ").strip()
if not topic:
    raise ValueError("Bạn cần nhập chủ đề.")

output_path = "/content/whiteboard_video.mp4"
command = [
    "pnpm", "--filter", "chalkboard", "start", "generate",
    topic,
    "--lang", "vi",
    "--aspect", "16:9",
    "--llm", "ollama",
    "--tts", "piper",
    "--no-images",
    "-o", output_path,
]
subprocess.run(command, cwd=REPO_DIR, check=True)

from google.colab import files
files.download(output_path)

## Ghi chú

- Mỗi phiên Colab mới cần chạy lại các cell cài đặt và tải model.
- Tải MP4 về sau khi render; file trong máy ảo có thể mất khi runtime bị xóa.
- Không nhập API key vào notebook. Cấu hình hiện tại dùng Ollama và Piper chạy trong Colab.
- Dự án gốc thuộc [Atharva-Kanherkar/chalkboard](https://github.com/Atharva-Kanherkar/chalkboard). Notebook này chỉ là bộ chạy Colab, không phải bản fork độc lập của toàn bộ mã nguồn.